# 01 - Exposed cohort (semaglutide / bariatric surgery)

**Runs in:** Truveta Studio notebook environment only (needs `truveta.study`, an
active `spark` session and `pyspark.pandas`).

**What this notebook does**

Builds the *exposed* delivery cohort: people with a delivery record that occurs
**after** a first semaglutide dispense (`source_type == "med"`) or **after** a
bariatric surgery procedure (`source_type == "surgery"`), together with every
covariate and outcome used downstream.

**Population snapshot:** `Delivery`

**Inputs:** Truveta snapshot tables + code set definitions (see `CONFIG` below).

**Outputs** (written to `study.get_output_path(fs=True)`):

| File | Contents |
|---|---|
| `test_t3.csv` | main exposed cohort, one row per person (analytic file) |
| `weights_test.csv` | all harmonised weight measurements (kg) |
| `medication_full.csv` | all semaglutide dispensing records (input to notebook 03) |
| `full_med_wo_weight.csv` | medication cohort **without** the weight requirement (sensitivity) |
| `test_zcodecount.csv` | number of pregnancy Z-code encounters per person |

**Run order:** 01 -> 02 -> 03 -> 04 -> 05.

> Sections marked **Appendix** at the bottom are superseded approaches kept for
> provenance. They are *not* part of the pipeline and should not be run.

## 1. Setup and configuration

Everything that a re-runner might need to change lives in this section.

In [ ]:
from truveta.study import Client, OutputMode, display_df

import re
import warnings
from datetime import datetime, timedelta
from typing import overload

import numpy as np
import pandas as pd
import pyspark.pandas as ps
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import Window

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# CONFIG - edit here, not below
# ============================================================================

POPULATION_TITLE = "Delivery"       # Truveta population to pull the snapshot from

# --- code set definitions -------------------------------------------------
CODESETS = {
    "delivery":            "/definitions/delivery",                 # conditionCodes / procedureCodes / ectopic_code / multiple_code
    "bariatric_surgery":   "/definitions/bariatric-surgery",
    "semaglutide":         "https://library.truveta.com/o/truveta/d/semaglutide-medication-code-set",
    "pregnancy_zcode":     "/definitions/pregnancy-zcode",
    "bmi":                 "https://library.truveta.com/o/truveta/d/tr-body-mass-index-bmi",
    "gestational_diabetes":"https://library.truveta.com/o/truveta-research/d/gestational-diabetes",
    "type2_diabetes":      "/definitions/type-2-diabetes",
    "gestational_htn":     "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension",
    "hypertension":        "/definitions/hypertension",
    "preeclampsia":        "/definitions/preeclampsia",
    "csection":            "/definitions/c-section",
    "stillbirth":          "https://library.truveta.com/o/truveta/d/stillbirth-code-set",
    "excessive_fetal_wt":  "/definitions/excessive-fetal-weight",
    "prenatal":            "/definitions/prenatal",
    "hyperlipidemia":      "https://library.truveta.com/o/truveta-research/d/hyperlipidemia",
    "osa":                 "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea",
    "depression":          "https://library.truveta.com/o/truveta-research/d/major-depression",
    "preterm_birth":       "/definitions/preterm-birth",             # appendix only
}

# ICD-10-CM code lists built inline (no library definition available)
ICD10 = {
    "intra_grow_restrict": ["O36.59", "Z36.4", "O36.5990", "O36.591",
                            "O36.592", "O36.593", "O36.599"],
    "primiparous":         ["Z34.00", "O09.611", "O09.511", "O09.512", "O09.513"],
    "multiparous":         ["O34.21", "O09.521", "O09.40", "O09.621",
                            "O09.41", "O09.42", "O09.43"],
    "prior_csection":      ["O34.212", "O34.211", "O34.21", "O34.219"],
    "prior_preterm_birth": ["Z87.51", "O09.21", "O09.211", "O09.212",
                            "O09.213", "O09.219"],
}

# LOINC codes used to pull body-weight measurements
WEIGHT_LOINC = ["58229-6", "18833-4", "29463-7", "3141-9", "3142-7",
                "8341-0", "8349-3", "8350-1", "8351-9"]

# --- study windows --------------------------------------------------------
SURGERY_CUTOFF   = "2017-01-01"   # earliest bariatric surgery kept
MED_CUTOFF       = "2021-01-01"   # earliest semaglutide dispense kept
DELIVERY_CUTOFF  = "2022-01-01"   # earliest delivery kept

GA_MIN_WEEKS     = 24             # plausible gestational age at delivery
GA_MAX_WEEKS     = 42
PRETERM_WEEKS    = 37             # < 37 completed weeks = preterm

ZCODE_MAX_LAG_DAYS = 300          # Z-code must be within this many days of delivery

PREPREG_WINDOW_WEEKS   = 12       # pre-pregnancy weight: +/- 12 weeks around LMP
PREDELIVERY_WINDOW_WKS = 4        # pre-delivery weight: 4 weeks before delivery
PRETREATMENT_WINDOW_DAYS = 183    # pre-treatment weight: 6 months before exposure

# --- plausible weight range (pounds) used when harmonising units ----------
LBS_LL, LBS_UL = 90, 700

# --- output file names ----------------------------------------------------
OUT_COHORT       = "/test_t3.csv"
OUT_WEIGHTS      = "/weights_test.csv"
OUT_MEDICATION   = "/medication_full.csv"
OUT_MED_NOWEIGHT = "/full_med_wo_weight.csv"
OUT_ZCODE_COUNT  = "/test_zcodecount.csv"

In [ ]:
# Use only one client statement; comment out whichever you are not using.
client = Client(output_mode=OutputMode.PandasOnSpark)
# client = Client(output_mode=OutputMode.PySpark)

study = client.get_study()
population = study.get_population(title=POPULATION_TITLE)
snapshot = population.get_latest_snapshot()

output_path_local = study.get_output_path(fs=True)

# Needed because several steps join pandas-on-Spark frames with different lineages.
ps.set_option("compute.ops_on_diff_frames", True)

snapshot.get_tables()

## 2. Shared helper functions

These are duplicated verbatim in every notebook of this study so that each
notebook can be uploaded and run on its own. The reference copy lives in
`src/truveta_helpers.py` - **if you change one, change them all.**

In [ ]:
@overload
def decode_concepts(df: pd.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> pd.DataFrame: ...
@overload
def decode_concepts(df: ps.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> ps.DataFrame: ...
@overload
def decode_concepts(df: DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> DataFrame: ...
def decode_concepts(df, drop_concepts: bool = True, columns: list[str] | None = None):
    """Replace every `*ConceptId` column with its human-readable concept name.

    Returns the same DataFrame type that was passed in.
    """
    def should_decode(col: str, dtype: str) -> bool:
        if columns:
            return col in columns
        return col.endswith("ConceptId") and str(dtype) in ("int", "int32", "float64", "float32")

    column_names = set(df.columns)

    def target_col(col: str) -> str:
        name = col.removesuffix("ConceptId")
        if name == col and drop_concepts:
            return name
        while name in column_names:
            name = f"{name}Name"
        return name

    def safe_name(name: str) -> str:
        while name in column_names:
            name = f"{name}_tmp"
        return name

    final_order: list[str] = []

    if isinstance(df, pd.DataFrame):
        lookup = None
        for col, dtype in zip(df.columns, df.dtypes):
            if should_decode(col, str(dtype)):
                name = target_col(col)
                column_names.add(name)
                if lookup is None:  # lazy lookup
                    lookup = (spark.sql("SELECT ConceptId, ConceptName FROM Concept")
                              .toPandas().set_index("ConceptId").ConceptName)
                df[name] = df[col].map(lookup)
                if drop_concepts and name != col:
                    df = df.drop(columns=[col])
                else:
                    final_order.append(col)
                final_order.append(name)
            else:
                final_order.append(col)
        return df[final_order]

    return_pandas = False
    if isinstance(df, ps.DataFrame):
        return_pandas = True
        df = df.to_spark()

    concepts_s = spark.sql("SELECT ConceptId, ConceptName FROM Concept").cache()
    for col, dtype in df.dtypes:
        if should_decode(col, str(dtype)):
            name = target_col(col)
            column_names.add(name)
            if col == name and drop_concepts:
                tmp_name = safe_name(col)
                df = (df.withColumnRenamed(col, tmp_name)
                        .join(concepts_s.withColumnRenamed("ConceptId", tmp_name)
                                        .withColumnRenamed("ConceptName", name),
                              on=tmp_name, how="left")
                        .drop(tmp_name))
            else:
                df = df.join(concepts_s.withColumnRenamed("ConceptId", col)
                                       .withColumnRenamed("ConceptName", name),
                             on=col, how="left")
                if drop_concepts:
                    df = df.drop(col)
                else:
                    final_order.append(col)
            final_order.append(name)
        else:
            final_order.append(col)
    df = df.select(final_order)
    return df.pandas_api() if return_pandas else df


def match_code(df, codes_df):
    """Decode concept ids and keep only rows whose `Code` belongs to the code set."""
    code_name = decode_concepts(df)
    case_names = codes_df.ConceptName.to_pandas().tolist()
    return code_name[code_name["Code"].isin(case_names)]

In [ ]:
def load_condition_data(snapshot, codeset_url=None, code_set=None, codes="codes",
                        table_name="Condition", view_name="tbl_index_condition",
                        concept_map_table="ConditionCodeConceptMap",
                        concept_map_key="CodeConceptMapId", verbose=True):
    """Load one clinical table filtered to a code set and decode it to code names.

    Supply either `codeset_url` (a prose definition) or a pre-built `code_set`.
    Returns `(matched_df, unique_person_count)`.
    """
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)
    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):",
              index_table["PersonId"].nunique())

    df = ps.sql(f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.*
        FROM {view_name} m
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
    """).to_pandas()

    matched_df = match_code(df, code_set)
    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df["PersonId"].nunique())

    return matched_df, matched_df["PersonId"].nunique()

In [ ]:
def mark_condition_in_pregnancy(condition_df, delivery_df, col_name):
    """Flag people whose condition was recorded between `estimated_LMP` and delivery."""
    merged = condition_df.merge(
        delivery_df[["PersonId", "estimated_LMP", "delivery_date"]],
        on="PersonId", how="left")

    merged["in_pregnancy"] = (
        (merged["RecordedDateTime"] >= merged["estimated_LMP"]) &
        (merged["RecordedDateTime"] <= merged["delivery_date"])
    )

    flagged_ids = merged.loc[merged["in_pregnancy"], "PersonId"].drop_duplicates()
    condition_flag = pd.DataFrame({"PersonId": flagged_ids, col_name: True})

    delivery_df = delivery_df.merge(condition_flag, on="PersonId", how="left")
    delivery_df[col_name] = delivery_df[col_name].fillna(False)
    return delivery_df


def condition_before_pregnancy(condition_df, delivery_df, condition_col_name):
    """Flag people with a condition recorded strictly before `estimated_LMP`."""
    merged = condition_df.merge(
        delivery_df[["PersonId", "estimated_LMP"]], on="PersonId", how="left")

    merged["pre_pregnancy"] = merged["RecordedDateTime"] < merged["estimated_LMP"]

    before_preg = merged[merged["pre_pregnancy"]].drop_duplicates(subset="PersonId")
    delivery_df[condition_col_name] = delivery_df["PersonId"].isin(
        before_preg["PersonId"].unique())
    return delivery_df

## 3. Exclusion code sets: ectopic pregnancy and multiple gestation

People with any ectopic-pregnancy or multiple-gestation record are removed from
the cohort (both are handled at step 6).

In [ ]:
# --- ectopic pregnancy (procedures + conditions) --------------------------
ectopics_code = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                            variable_name="ectopic_code")

ectopics_pro = snapshot.load_filtered_table("Procedure", ectopics_code,
                                            view_name="tbl_ectopics_pro")
print("ectopic procedures, unique persons:", ectopics_pro["PersonId"].nunique())

ectopics_con = snapshot.load_filtered_table("Condition", ectopics_code,
                                            view_name="tbl_ectopics_con")
print("ectopic conditions, unique persons:", ectopics_con["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.*
               FROM tbl_ectopics_pro p
               JOIN ProcedureCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
ectopics_pro = match_code(df, ectopics_code)

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, pm.*
               FROM tbl_ectopics_con p
               JOIN ConditionCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
ectopics_con = match_code(df, ectopics_code)

ectopics = pd.concat([ectopics_pro, ectopics_con])
print("ectopic (combined), unique persons:", ectopics.PersonId.nunique())

In [ ]:
# --- multiple gestation ---------------------------------------------------
multiple_code = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                            variable_name="multiple_code")

multiple_df = snapshot.load_filtered_table("Condition", multiple_code,
                                           view_name="tbl_multiple")
print("multiple gestation, unique persons:", multiple_df["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, pm.*
               FROM tbl_multiple p
               JOIN ConditionCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
multiple_df = match_code(df, multiple_code)

## 4. Exposure ascertainment

### 4.1 Bariatric surgery group

One row per person per surgery date, from `SURGERY_CUTOFF` onwards.

In [ ]:
surgerycodes_df = snapshot.codeset_from_prose(url=CODESETS["bariatric_surgery"],
                                              variable_name="codes")
index_surgery = snapshot.load_filtered_table("Procedure", surgerycodes_df,
                                             view_name="tbl_index_surgery")
print("bariatric surgery, unique persons:", index_surgery["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.*
               FROM tbl_index_surgery p
               JOIN ProcedureCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
procedure_full = match_code(df, surgerycodes_df)

# Procedures sometimes only carry RecordedDateTime; fall back to it.
procedure_full["StartDateTime"] = procedure_full["StartDateTime"].fillna(
    procedure_full["RecordedDateTime"])
procedure_full = procedure_full.drop(columns=["RecordedDateTime"]).dropna()

procedure_full = (procedure_full
                  .sort_values(["PersonId", "StartDateTime"])
                  .reset_index(drop=True))
procedure_full["StartDateTime"] = procedure_full["StartDateTime"].dt.date
procedure_full = procedure_full[
    procedure_full["StartDateTime"] >= datetime.strptime(SURGERY_CUTOFF, "%Y-%m-%d").date()]
procedure_full = (procedure_full
                  .drop_duplicates(subset=["PersonId", "StartDateTime"], keep="first")
                  .reset_index(drop=True))

print("after cutoff + dedup, unique persons:", procedure_full["PersonId"].nunique())

### 4.2 Semaglutide medication group

Uses `MedicationDispense` (fill records), from `MED_CUTOFF` onwards.
`medication_full_copy` keeps *every* fill; `medication_full` is one row per
person per dispense date and is what drives the cohort definition.

In [ ]:
medcodes_df = snapshot.codeset_from_prose(url=CODESETS["semaglutide"],
                                          variable_name="codes")
index_med = snapshot.load_filtered_table("MedicationDispense", medcodes_df,
                                         view_name="tbl_index_med")
print("semaglutide dispense, unique persons:", index_med["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.DaysSupply, m.DispenseDateTime, pm.*
               FROM tbl_index_med m
               JOIN MedicationCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
medication_full = match_code(df, medcodes_df)

print("rows with any missing value:", len(medication_full[medication_full.isna().any(axis=1)]))
medication_full = medication_full.dropna()
print("rows / persons after dropna:", len(medication_full), medication_full.PersonId.nunique())

In [ ]:
medication_full = (medication_full
                   .sort_values(["PersonId", "DispenseDateTime"])
                   .reset_index(drop=True))
medication_full["DispenseDateTime"] = medication_full["DispenseDateTime"].dt.date
medication_full = medication_full[
    medication_full["DispenseDateTime"] >= datetime.strptime(MED_CUTOFF, "%Y-%m-%d").date()]

medication_full_copy = medication_full.copy()   # all fills, kept for reference
medication_full = (medication_full
                   .drop_duplicates(subset=["PersonId", "DispenseDateTime"], keep="first")
                   .reset_index(drop=True))

print("after cutoff + dedup, unique persons:", medication_full["PersonId"].nunique())

### 4.3 Overlap between exposure groups

People with **both** a bariatric surgery and a semaglutide dispense are removed
so the two exposure arms stay mutually exclusive.

In [ ]:
both_med_surg = procedure_full.merge(medication_full, on="PersonId", how="inner")
print("both medication and surgery:", both_med_surg.PersonId.nunique())

for label, left in [("surgery", procedure_full), ("medication", medication_full)]:
    print(f"{label} x ectopic  :", left.merge(ectopics, on="PersonId", how="inner").PersonId.nunique())
    print(f"{label} x multiple :", left.merge(multiple_df, on="PersonId", how="inner").PersonId.nunique())

## 5. Body weight measurements

Weights come from both `LabResult` and `Observation`. Units in the source data
are inconsistent, so the block below reproduces the Truveta Enablement-study
approach: use the recorded unit where it is known, otherwise infer the unit from
the magnitude of the value, then convert everything to pounds and finally kg.
Values outside `LBS_LL`-`LBS_UL` pounds are treated as implausible and dropped.

In [ ]:
concept = snapshot.load_sql_table("SELECT * FROM Concept",
                                  view_name="tbl_concept")[["ConceptId", "ConceptName"]]

weight_codes_s = snapshot.codeset("LOINC", "selfAndDescendants", *WEIGHT_LOINC)
study.create_view(weight_codes_s, view_name="tbl_weight_codes_s")

WEIGHT_COLS = ["PersonId", "EffectiveDateTime", "RecordedDateTime",
               "NormalizedValueConceptId", "NormalizedValueNumeric",
               "NormalizedValueUOMConceptId", "StatusConceptId", "EncounterId"]

lab_weights = snapshot.load_filtered_table("LabResult", weight_codes_s,
                                           view_name="tbl_lab_weights")[WEIGHT_COLS]
obs_weights = snapshot.load_filtered_table("Observation", weight_codes_s,
                                           view_name="tbl_obs_weights")[WEIGHT_COLS]

all_weights = ps.concat([obs_weights, lab_weights], ignore_index=True)
display_df(all_weights)

In [ ]:
# Decode the three concept columns we care about.
for id_col, name_col in [("NormalizedValueConceptId", "NormalizedValueConcept"),
                         ("NormalizedValueUOMConceptId", "NormalizedValueUOMConcept"),
                         ("StatusConceptId", "StatusConcept")]:
    all_weights = (all_weights
                   .merge(concept, how="left", left_on=id_col, right_on="ConceptId")
                   .rename(columns={"ConceptName": name_col})
                   .drop(columns=["ConceptId"]))

In [ ]:
# Units that cannot possibly be a body weight.
EXCLUDED_UNITS = [
    "per liter", "per meter", "billion per liter", "centimeter", "degree",
    "foot (US)", "heart beats per minute", "inches", "liter", "liter per minute",
    "lumen", "meter", "millimeter mercury column", "millivolt", "minute",
    "per hour", "percent", "second", "week", "Each", "Inches",
    "inch (international)", "each",
]
all_weights = all_weights[~all_weights["NormalizedValueUOMConcept"].isin(EXCLUDED_UNITS)]

# Collapse the several flavours of "missing" into a single label.
all_weights["NormalizedValueUOMConcept"] = all_weights["NormalizedValueUOMConcept"].replace({
    "No Information": "unknown",
    "Field has not been mapped": "unknown",
    "Field is not present in source": "unknown",
    "Invalid": "unknown",
})

all_weights.groupby("NormalizedValueUOMConcept").size().reset_index()

In [ ]:
POUND_UNITS = ["pound (US and British)", "pound (US)", "pound (apothecary)"]

all_weights["NormalizedValueNumeric"] = all_weights["NormalizedValueNumeric"].astype(float)
all_weights["UOM_assumed"] = np.nan

# 1. Known unit -> use it as-is (all pound variants collapse to "pound").
all_weights.loc[all_weights["NormalizedValueUOMConcept"].isin(POUND_UNITS), "UOM_assumed"] = "pound"
all_weights.loc[
    (all_weights["NormalizedValueUOMConcept"] != "unknown") &
    (~all_weights["NormalizedValueUOMConcept"].isin(POUND_UNITS)),
    "UOM_assumed"] = all_weights["NormalizedValueUOMConcept"]

# 2. Unknown unit -> infer from magnitude.
all_weights.loc[
    (all_weights["NormalizedValueNumeric"] > LBS_LL * 453.6) &
    (all_weights["NormalizedValueNumeric"] <= LBS_UL * 453.6) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "gram"

all_weights.loc[
    (all_weights["NormalizedValueNumeric"] > LBS_LL * 16) &
    (all_weights["NormalizedValueNumeric"] <= LBS_UL * 16) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "ounce (avoirdupois)"

all_weights.loc[
    (all_weights["NormalizedValueNumeric"] < 125) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "kilogram"

# 3. Convert everything to pounds, then to kg.
all_weights["pounds"] = np.nan
all_weights.loc[all_weights["UOM_assumed"] == "pound", "pounds"] = all_weights["NormalizedValueNumeric"]
all_weights.loc[all_weights["UOM_assumed"] == "ounce (avoirdupois)", "pounds"] = all_weights["NormalizedValueNumeric"] / 16
all_weights.loc[all_weights["UOM_assumed"] == "kilogram", "pounds"] = all_weights["NormalizedValueNumeric"] * 2.205
all_weights.loc[all_weights["UOM_assumed"] == "gram", "pounds"] = all_weights["NormalizedValueNumeric"] / 453.6

# 4. Drop implausible values and derive kg.
all_weights.loc[(all_weights["pounds"] < LBS_LL) | (all_weights["pounds"] > LBS_UL), "pounds"] = np.nan
all_weights["kg"] = all_weights["pounds"] / 2.205

all_weights.head()

## 6. Delivery records and the index delivery

Delivery is captured from both `Condition` and `Procedure` code sets. Records
are pooled, deduplicated to one row per person per date, filtered to deliveries
on/after `DELIVERY_CUTOFF`, and then restricted to the **first** delivery that
occurs after the person's exposure (surgery or medication) date.

In [ ]:
# --- delivery conditions --------------------------------------------------
delivery_concode = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                               variable_name="conditionCodes")
delivery_con = snapshot.load_filtered_table("Condition", delivery_concode,
                                            view_name="tbl_index_delivery_con")
print("delivery conditions, unique persons:", delivery_con["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.RecordedDateTime, pm.*
               FROM tbl_index_delivery_con m
               JOIN ConditionCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
delivery_con = match_code(df, delivery_concode)
print("rows / persons:", len(delivery_con), delivery_con.PersonId.nunique())

# --- delivery procedures --------------------------------------------------
delivery_procode = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                               variable_name="procedureCodes")
delivery_pro = snapshot.load_filtered_table("Procedure", delivery_procode,
                                            view_name="tbl_index_delivery_pro")
print("delivery procedures, unique persons:", delivery_pro["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.StartDateTime, pm.*
               FROM tbl_index_delivery_pro m
               JOIN ProcedureCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
delivery_pro = match_code(df, delivery_procode)
delivery_pro.head()

In [ ]:
# Pool conditions and procedures on a common date column.
delivery_pro = delivery_pro.rename(columns={"StartDateTime": "RecordedDateTime"})
combined_df = pd.concat([delivery_con, delivery_pro])
combined_df["RecordedDateTime"] = pd.to_datetime(combined_df["RecordedDateTime"])
combined_df = combined_df.sort_values(["PersonId", "RecordedDateTime"]).reset_index(drop=True)
combined_df["RecordedDateTime"] = combined_df["RecordedDateTime"].dt.date
print("delivery records / persons:", len(combined_df), combined_df.PersonId.nunique())

In [ ]:
# One row per person-date, then apply the exclusions.
combined_df_cleaned = (combined_df
                       .drop_duplicates(subset=["PersonId", "RecordedDateTime"], keep="first")
                       .reset_index(drop=True))

combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned["PersonId"].isin(both_med_surg["PersonId"])]
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned["PersonId"].isin(ectopics["PersonId"])]
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned["PersonId"].isin(multiple_df["PersonId"])]
print("after exclusions:", len(combined_df_cleaned), combined_df_cleaned.PersonId.nunique())

combined_df_cleaned = combined_df_cleaned.dropna()
combined_df_cleaned = combined_df_cleaned[
    combined_df_cleaned["RecordedDateTime"] >= datetime.strptime(DELIVERY_CUTOFF, "%Y-%m-%d").date()
].reset_index(drop=True)
print("after delivery cutoff:", len(combined_df_cleaned), combined_df_cleaned.PersonId.nunique())

In [ ]:
# Attach exposure dates and keep the first delivery that follows an exposure.
procedure_full = procedure_full.rename(columns={"StartDateTime": "surgery_date"})
medication_full = medication_full.rename(columns={"DispenseDateTime": "med_date"})

merged = (combined_df_cleaned
          .merge(procedure_full[["PersonId", "surgery_date"]], on="PersonId", how="left")
          .merge(medication_full[["PersonId", "med_date"]], on="PersonId", how="left"))

merged["is_after_surgery"] = merged["RecordedDateTime"] > merged["surgery_date"]
merged["is_after_med"] = merged["RecordedDateTime"] > merged["med_date"]


def get_source_type(row):
    """Label the delivery by which exposure preceded it (surgery takes priority)."""
    if row["is_after_surgery"]:
        return "surgery"
    if row["is_after_med"]:
        return "med"
    return None


merged["source_type"] = merged.apply(get_source_type, axis=1)

filtered = merged[merged["source_type"].notna()].sort_values(["PersonId", "RecordedDateTime"])
first_events = filtered.groupby("PersonId").first().reset_index()

# `event_date` = the exposure date that qualified this delivery.
first_events["event_date"] = first_events["surgery_date"].fillna(first_events["med_date"])

delivery_df = first_events[["PersonId", "RecordedDateTime", "Code",
                            "source_type", "event_date"]].copy()

print("index deliveries:", len(first_events), first_events.PersonId.nunique())
print("surgery persons:", merged[merged.source_type == "surgery"].PersonId.nunique(),
      "| med persons:", merged[merged.source_type == "med"].PersonId.nunique())
delivery_df.head()

In [ ]:
# Drop deliveries whose *delivery code text itself* says preterm/premature -
# gestational age is derived from Z-codes below and these records would be
# double-counted / mis-dated.
def is_preterm_related(label):
    label = label.lower()
    return "preterm" in label or "premature" in label


coded_preterm = delivery_df[delivery_df["Code"].apply(is_preterm_related)].copy()
print("preterm-coded deliveries:", len(coded_preterm), coded_preterm.PersonId.nunique())

print("before:", len(delivery_df), delivery_df.PersonId.nunique())
delivery_df = delivery_df[~delivery_df["PersonId"].isin(coded_preterm["PersonId"])]
print("after :", len(delivery_df), delivery_df.PersonId.nunique())

## 7. Gestational age from pregnancy Z-codes

ICD-10 `Z3A.*` codes state the week of gestation at an encounter. For each
person we take the Z-code encounter **closest to delivery** (and no more than
`ZCODE_MAX_LAG_DAYS` before it), back-calculate the estimated last menstrual
period (LMP), and derive gestational age at delivery.

`zcode_count` (number of qualifying Z-code encounters) is retained as a crude
proxy for prenatal-care engagement / data density.

In [ ]:
zcodecode = snapshot.codeset_from_prose(url=CODESETS["pregnancy_zcode"],
                                        variable_name="zcodes")
zcode = snapshot.load_filtered_table("Condition", zcodecode, view_name="tbl_index_zcodes")
print("Z-code, unique persons:", zcode["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.RecordedDateTime, pm.*
               FROM tbl_index_zcodes m
               JOIN ConditionCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
zcode = match_code(df, zcodecode)
print("rows / persons:", len(zcode), zcode.PersonId.nunique())

In [ ]:
# Drop Z-codes that carry no usable week, then parse the week number out of the
# code description and keep the highest week per person-encounter.
mask_remove = zcode["Code"].str.lower().isin([
    "weeks of gestation of pregnancy not specified",
    "less than 8 weeks gestation of pregnancy",
])
zcode = zcode.loc[~mask_remove].copy()

zcode["gestational_week_zcode"] = zcode["Code"].str.extract(r"(\d+)", expand=False).astype("float")

zcode = (zcode
         .sort_values(["PersonId", "RecordedDateTime", "gestational_week_zcode"])
         .groupby(["PersonId", "RecordedDateTime"], as_index=False)
         .tail(1))

zcode.gestational_week_zcode.describe()

In [ ]:
zcode = zcode.rename(columns={"RecordedDateTime": "zcodetime", "Code": "zcode"})

deliv_zcode = delivery_df.merge(zcode, on="PersonId", how="inner")
print("delivery x zcode rows / persons:", len(deliv_zcode), deliv_zcode.PersonId.nunique())

deliv_zcode["RecordedDateTime"] = pd.to_datetime(deliv_zcode["RecordedDateTime"], errors="coerce")
deliv_zcode["zcodetime"] = pd.to_datetime(deliv_zcode["zcodetime"], errors="coerce")

# Z-code must precede delivery and be within the lag window.
deliv_zcode = deliv_zcode[deliv_zcode["zcodetime"] <= deliv_zcode["RecordedDateTime"]].copy()
deliv_zcode["diff_days"] = (deliv_zcode["RecordedDateTime"] - deliv_zcode["zcodetime"]) / pd.Timedelta(days=1)
deliv_zcode = deliv_zcode[deliv_zcode["diff_days"] <= ZCODE_MAX_LAG_DAYS].copy()

# One (highest-week) Z-code per calendar day, then the one closest to delivery.
deliv_zcode["zcodetime_date"] = deliv_zcode["zcodetime"].dt.date
deliv_zcode = (deliv_zcode
               .sort_values(["PersonId", "RecordedDateTime", "zcodetime_date", "gestational_week_zcode"])
               .groupby(["PersonId", "RecordedDateTime", "zcodetime_date"], as_index=False)
               .tail(1))

idx = deliv_zcode.groupby(["PersonId", "RecordedDateTime"])["diff_days"].idxmin()
closest_df = deliv_zcode.loc[idx].copy()
print("closest z-code per delivery:", len(closest_df), closest_df.PersonId.nunique())

zcode_counts = (deliv_zcode
                .groupby(["PersonId", "RecordedDateTime"])
                .size()
                .reset_index(name="zcode_count"))
print("zcode_counts:", zcode_counts.shape, zcode_counts.PersonId.nunique())

In [ ]:
delivery_df = closest_df.copy()

# LMP = Z-code encounter date minus the stated gestational weeks.
delivery_df["estimated_LMP"] = (
    delivery_df["zcodetime"] - pd.to_timedelta(delivery_df["gestational_week_zcode"] * 7, unit="D"))

delivery_df["gestational_week"] = (
    (delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"]) / pd.Timedelta(days=7))
delivery_df["gestational_age_days_at_delivery"] = (
    delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"]).dt.days

# Keep only physiologically plausible gestational ages.
delivery_df = delivery_df[(delivery_df["gestational_week"] >= GA_MIN_WEEKS) &
                          (delivery_df["gestational_week"] <= GA_MAX_WEEKS)].copy()
print(delivery_df.gestational_week.describe())

delivery_df["preterm"] = (delivery_df["gestational_week"] < PRETERM_WEEKS).astype(int)
print(delivery_df["preterm"].value_counts())

delivery_df = delivery_df.merge(zcode_counts[["PersonId", "zcode_count"]],
                                on="PersonId", how="inner")
print("cohort shape:", delivery_df.shape)

In [ ]:
mode_week = delivery_df["gestational_week"].round().mode()[0]
plt.figure()
delivery_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Exposed cohort: gestational age at delivery")
plt.legend()
plt.show()

In [ ]:
# Medication-only cohort *before* the weight requirement is applied.
# Exported as a sensitivity sample (see section 13).
full_med = delivery_df[delivery_df["source_type"] == "med"].copy()
print("medication cohort before weight requirement:", len(full_med))

## 8. Attaching weights and computing gestational weight gain

Three weights are extracted per person:

| Variable | Window |
|---|---|
| `prepreg_weight` | closest measurement within +/- `PREPREG_WINDOW_WEEKS` of the estimated LMP |
| `predelivery_weight` | closest measurement in the `PREDELIVERY_WINDOW_WKS` weeks before delivery |
| `pretreatment_weight` | closest measurement in the `PRETREATMENT_WINDOW_DAYS` days before the exposure date |

`gestation_weight` (gestational weight gain, kg) = `predelivery_weight - prepreg_weight`,
and only people with **both** anchoring weights are kept.

In [ ]:
all_weights_df = all_weights[["PersonId", "RecordedDateTime", "pounds", "kg"]].copy()
all_weights_df["RecordedDateTime"] = all_weights_df["RecordedDateTime"].dt.date
all_weights_df = all_weights_df.dropna()

all_weights_df_pd = all_weights_df.to_pandas()
print("weight measurements:", all_weights_df_pd.shape)
all_weights_df_pd.head()

In [ ]:
def add_weight_info(delivery_df, all_weights_df):
    """Attach pre-pregnancy, pre-delivery and pre-treatment weights to each delivery.

    Also computes `months_between_event_and_conception` (positive = LMP after the
    exposure date). Rows without a pre-treatment weight are dropped, since the
    exposed cohort is defined relative to treatment initiation.
    """
    delivery_df = delivery_df.copy()
    all_weights_df = all_weights_df.copy()

    for col in ["RecordedDateTime", "event_date", "estimated_LMP"]:
        delivery_df[col] = pd.to_datetime(delivery_df[col], errors="coerce")
    all_weights_df["RecordedDateTime"] = pd.to_datetime(
        all_weights_df["RecordedDateTime"], errors="coerce")

    delivery_df = delivery_df.dropna(
        subset=["PersonId", "RecordedDateTime", "event_date", "estimated_LMP"])
    all_weights_df = all_weights_df.dropna(subset=["PersonId", "RecordedDateTime", "kg"])

    weight_around_lmp, has_weight_around_lmp = [], []
    weight_before_del, has_weight_before_del = [], []
    latest_weight_before_event, months_between_event_and_lmp = [], []

    for row in delivery_df.itertuples(index=False):
        person_weights = all_weights_df[all_weights_df["PersonId"] == row.PersonId]
        lmp = pd.Timestamp(row.estimated_LMP)
        delivery = pd.Timestamp(row.RecordedDateTime)
        event_date = pd.Timestamp(row.event_date)

        # --- pre-pregnancy: closest measurement within +/- N weeks of LMP -----
        con_window = person_weights[
            ((person_weights["RecordedDateTime"] >= lmp - timedelta(weeks=PREPREG_WINDOW_WEEKS)) &
             (person_weights["RecordedDateTime"] < lmp)) |
            ((person_weights["RecordedDateTime"] > lmp) &
             (person_weights["RecordedDateTime"] <= lmp + timedelta(weeks=PREPREG_WINDOW_WEEKS)))]
        if not con_window.empty:
            nearest = con_window.loc[(con_window["RecordedDateTime"] - lmp).abs().idxmin()]
            weight_around_lmp.append(nearest["kg"])
            has_weight_around_lmp.append(True)
        else:
            weight_around_lmp.append(pd.NA)
            has_weight_around_lmp.append(False)

        # --- pre-delivery: closest measurement in the N weeks before delivery -
        del_window = person_weights[
            (person_weights["RecordedDateTime"] >= delivery - timedelta(weeks=PREDELIVERY_WINDOW_WKS)) &
            (person_weights["RecordedDateTime"] <= delivery)]
        if not del_window.empty:
            nearest_del = del_window.loc[(del_window["RecordedDateTime"] - delivery).abs().idxmin()]
            weight_before_del.append(nearest_del["kg"])
            has_weight_before_del.append(True)
        else:
            weight_before_del.append(pd.NA)
            has_weight_before_del.append(False)

        # --- pre-treatment: closest measurement in the N days before exposure -
        before_event = person_weights[
            (person_weights["RecordedDateTime"] >= event_date - timedelta(days=PRETREATMENT_WINDOW_DAYS)) &
            (person_weights["RecordedDateTime"] <= event_date)]
        if not before_event.empty:
            nearest_before_event = before_event.loc[
                (before_event["RecordedDateTime"] - event_date).abs().idxmin()]
            latest_weight_before_event.append(nearest_before_event["kg"])
        else:
            latest_weight_before_event.append(pd.NA)

        months_between_event_and_lmp.append(round((lmp - event_date).days / 30.44, 1))

    delivery_df["has_prepreg_weight"] = has_weight_around_lmp
    delivery_df["prepreg_weight"] = weight_around_lmp
    delivery_df["has_predelivery_weight"] = has_weight_before_del
    delivery_df["predelivery_weight"] = weight_before_del
    delivery_df["pretreatment_weight"] = latest_weight_before_event
    delivery_df["months_between_event_and_conception"] = months_between_event_and_lmp

    return delivery_df[delivery_df["pretreatment_weight"].notna()].copy()

In [ ]:
delivery_df_pd = add_weight_info(delivery_df, all_weights_df_pd)
print(delivery_df_pd["has_prepreg_weight"].value_counts())
print(delivery_df_pd["has_predelivery_weight"].value_counts())

In [ ]:
# Require both anchoring weights, then derive gestational weight gain.
delivery_df_pd["has_both_weights"] = (
    delivery_df_pd["has_prepreg_weight"] & delivery_df_pd["has_predelivery_weight"])
delivery_df_pd = delivery_df_pd[delivery_df_pd["has_both_weights"]].copy()

delivery_df_pd["gestation_weight"] = (
    delivery_df_pd["predelivery_weight"] - delivery_df_pd["prepreg_weight"])

print("cohort with GWG:", delivery_df_pd.shape)
print("median GWG (kg):", delivery_df_pd.gestation_weight.median())
print(delivery_df_pd.source_type.value_counts())

In [ ]:
# Persist the Z-code counts for this cohort (used by notebook 05).
zcode_count_df = delivery_df_pd[["PersonId", "zcode_count"]].copy()
zcode_count_df.to_csv(output_path_local + OUT_ZCODE_COUNT, index=False)

## 9. BMI

Pre-pregnancy BMI is the measurement closest in time to the estimated LMP;
`nearDeliveryBMI` and `preTreatmentBMI` are the measurements closest to the
delivery date.

> Note: `preTreatmentBMI` is anchored on the **delivery** date in the original
> analysis, not on `event_date`. That behaviour is preserved here so results
> reproduce; treat `preTreatmentBMI` and `nearDeliveryBMI` as the same quantity.

In [ ]:
bmi_df = snapshot.codeset_from_prose(url=CODESETS["bmi"], variable_name="codes")
index_bmi_df = snapshot.load_filtered_table("Observation", bmi_df, view_name="tbl_index_bmi")
print("BMI observations, unique persons:", index_bmi_df["PersonId"].nunique())
index_bmi_df_lab = snapshot.load_filtered_table("LabResult", bmi_df, view_name="tbl_index_bmilab")

index_bmi_df["RecordedDateTime"] = pd.to_datetime(index_bmi_df["RecordedDateTime"]).astype("datetime64[ns]")
index_bmi_df = ps.from_pandas(index_bmi_df)
index_bmi_df = ps.concat([index_bmi_df, index_bmi_df_lab])

index_bmi_df = (index_bmi_df[["NormalizedValueNumeric", "PersonId", "RecordedDateTime"]]
                .sort_values(["PersonId", "RecordedDateTime"])
                .reset_index(drop=True)
                .dropna(subset=["NormalizedValueNumeric", "RecordedDateTime"])
                .drop_duplicates(subset=["PersonId", "RecordedDateTime"], keep="first")
                .to_pandas())
index_bmi_df["RecordedDateTime"] = ps.to_datetime(index_bmi_df["RecordedDateTime"])
print("persons with BMI:", index_bmi_df.PersonId.nunique())

In [ ]:
delivery_df_pd = delivery_df_pd.rename(columns={"RecordedDateTime": "delivery_date"})

index_bmi_df["NormalizedValueNumeric"] = index_bmi_df["NormalizedValueNumeric"].astype(float)
index_bmi_df = index_bmi_df.dropna(subset=["NormalizedValueNumeric"])
index_bmi_df = index_bmi_df[index_bmi_df["NormalizedValueNumeric"] > 10.0]   # drop impossible BMIs

merged_df = pd.merge(index_bmi_df, delivery_df_pd, on="PersonId")
for col in ["RecordedDateTime", "estimated_LMP", "delivery_date"]:
    merged_df[col] = pd.to_datetime(merged_df[col])


def closest_bmi(df, anchor_col, out_name):
    """BMI measurement closest in time to `anchor_col`, one row per person."""
    tmp = df.copy()
    tmp["TimeDiff"] = (tmp[anchor_col] - tmp["RecordedDateTime"]).abs()
    return (tmp.sort_values(["PersonId", "TimeDiff"])
               .drop_duplicates("PersonId", keep="first")[["PersonId", "NormalizedValueNumeric"]]
               .rename(columns={"NormalizedValueNumeric": out_name}))


pre_bmi_df = closest_bmi(merged_df, "estimated_LMP", "PrePregnancyBMI")
post_bmi_df = closest_bmi(merged_df, "delivery_date", "nearDeliveryBMI")
pretreat_df = closest_bmi(merged_df, "delivery_date", "preTreatmentBMI")

delivery_df_pd = (delivery_df_pd
                  .merge(pre_bmi_df, on="PersonId", how="left")
                  .merge(post_bmi_df, on="PersonId", how="left"))
delivery_df_pd.nearDeliveryBMI.describe()

## 10. Outcome and covariate code sets

In [ ]:
gest_diabet, _ = load_condition_data(snapshot, codeset_url=CODESETS["gestational_diabetes"],
                                     view_name="tbl_index_gest_diabet")

type2_diabet, _ = load_condition_data(snapshot, codeset_url=CODESETS["type2_diabetes"],
                                      view_name="tbl_index_type2_diabet")

gest_hyper, _ = load_condition_data(snapshot, codeset_url=CODESETS["gestational_htn"],
                                    view_name="tbl_index_gest_hyper")

hypertension, _ = load_condition_data(snapshot, codeset_url=CODESETS["hypertension"],
                                      codes="conditionCodes", view_name="tbl_index_hyer")

preeclampsia, _ = load_condition_data(snapshot, codeset_url=CODESETS["preeclampsia"],
                                      view_name="tbl_index_preeclampsia")

stillbirth, _ = load_condition_data(snapshot, codeset_url=CODESETS["stillbirth"],
                                    view_name="tbl_index_sb")

excessive_fetal_weight, _ = load_condition_data(snapshot, codeset_url=CODESETS["excessive_fetal_wt"],
                                                view_name="tbl_index_efw")

Prenatal, _ = load_condition_data(snapshot, codeset_url=CODESETS["prenatal"],
                                  codes="Prenatal", view_name="tbl_index_Prenatal")

hyperlipidemia, _ = load_condition_data(snapshot, codeset_url=CODESETS["hyperlipidemia"],
                                        view_name="tbl_index_hyerlip")

osa, _ = load_condition_data(snapshot, codeset_url=CODESETS["osa"],
                             codes="osaCodes", view_name="tbl_index_osa")

depression, _ = load_condition_data(snapshot, codeset_url=CODESETS["depression"],
                                    view_name="tbl_index_mdep")

In [ ]:
# C-section is a procedure, so it needs the StartDateTime fallback treatment.
csection_code = snapshot.codeset_from_prose(url=CODESETS["csection"], variable_name="codes")
index_c = snapshot.load_filtered_table("Procedure", csection_code, view_name="tbl_index_c")
print("c-section, unique persons:", index_c["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.*
               FROM tbl_index_c p
               JOIN ProcedureCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
csection = match_code(df, csection_code)
csection["StartDateTime"] = csection["StartDateTime"].fillna(csection["RecordedDateTime"])
csection = (csection.drop(columns=["RecordedDateTime"])
                    .rename(columns={"StartDateTime": "RecordedDateTime"}))
csection.head()

In [ ]:
# Code sets built from explicit ICD-10-CM lists.
intra_grow_restrict, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["intra_grow_restrict"]),
    view_name="tbl_index_intra_grow_restrict")

primiparous, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["primiparous"]),
    view_name="tbl_index_primiparous")

multiparous, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["multiparous"]),
    view_name="tbl_index_multiparous")

prior_Csection, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["prior_csection"]),
    view_name="tbl_index_prior_Csection")

Prior_Preterm_Birth, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["prior_preterm_birth"]),
    view_name="tbl_index_Prior_Preterm_Birth")

## 11. Building the analytic variables

`delivery_df_t3` is the main cohort (weight requirement applied).
`full_med` is the medication-only sensitivity cohort (no weight requirement) -
the same derivations are applied to both.

In [ ]:
full_med = full_med.rename(columns={"RecordedDateTime": "delivery_date"})

# --- conditions recorded *during* pregnancy -------------------------------
IN_PREGNANCY_FLAGS = [
    (gest_diabet, "gest_diabet"),
    (gest_hyper, "gest_hyper"),
    (preeclampsia, "preeclampsia"),
    (csection, "csection"),
    (excessive_fetal_weight, "excessive_fetal_weight"),
    (intra_grow_restrict, "intra_grow_restrict"),
    (Prenatal, "obstetric_care"),
]

delivery_df_t3 = delivery_df_pd
for condition_df, col in IN_PREGNANCY_FLAGS:
    delivery_df_t3 = mark_condition_in_pregnancy(condition_df, delivery_df_t3, col)
    full_med = mark_condition_in_pregnancy(condition_df, full_med, col)

delivery_df_t3.head()

In [ ]:
# --- conditions recorded *before* pregnancy -------------------------------
BEFORE_PREGNANCY_FLAGS = [
    (type2_diabet, "t2d_before_pregnancy"),
    (hypertension, "hyper_before_pregnancy"),
    (hyperlipidemia, "hyperlipid"),
    (osa, "osa"),
    (depression, "depression"),
]

for condition_df, col in BEFORE_PREGNANCY_FLAGS:
    delivery_df_t3 = condition_before_pregnancy(condition_df, delivery_df_t3, col)
    full_med = condition_before_pregnancy(condition_df, full_med, col)

print(delivery_df_t3.t2d_before_pregnancy.value_counts())
print(delivery_df_t3.hyper_before_pregnancy.value_counts())

In [ ]:
# --- obstetric history (any time, not restricted to before pregnancy) -----
for frame in ("delivery_df_t3", "full_med"):
    pass  # both handled explicitly below for readability

delivery_df_t3["prior_Csection"] = delivery_df_t3["PersonId"].isin(prior_Csection["PersonId"])
delivery_df_t3["Prior_Preterm_Birth"] = delivery_df_t3["PersonId"].isin(Prior_Preterm_Birth["PersonId"])
full_med["prior_Csection"] = full_med["PersonId"].isin(prior_Csection["PersonId"])
full_med["Prior_Preterm_Birth"] = full_med["PersonId"].isin(Prior_Preterm_Birth["PersonId"])

# --- parity ---------------------------------------------------------------
for frame in (delivery_df_t3, full_med):
    frame["parity"] = "Unknown"
    frame.loc[frame["PersonId"].isin(primiparous["PersonId"]), "parity"] = "Primiparous"
    frame.loc[frame["PersonId"].isin(multiparous["PersonId"]), "parity"] = "Multiparous"

# --- derived delivery / infant labels -------------------------------------
for frame in (delivery_df_t3, full_med):
    frame["delivery_type"] = np.where(frame["csection"] == True, "C-section", "Vaginal")
    frame["infant_gest_age_class"] = np.where(
        frame["excessive_fetal_weight"] == True, "Excessive", "Average")

print(delivery_df_t3.delivery_type.value_counts())
print(delivery_df_t3.infant_gest_age_class.value_counts())

In [ ]:
# --- incident (pregnancy-onset) conditions --------------------------------
# "Gestational" only counts if the person did not already carry the chronic
# diagnosis before pregnancy.
for frame in (delivery_df_t3, full_med):
    frame["gest_diabetes_no_prior_t2d"] = frame["gest_diabet"] & ~frame["t2d_before_pregnancy"]
    frame["gest_hyper_no_prior_hyper"] = frame["gest_hyper"] & ~frame["hyper_before_pregnancy"]
    frame["preeclampsia_no_prior_hyper"] = frame["preeclampsia"] & ~frame["hyper_before_pregnancy"]

print(delivery_df_t3.gest_diabetes_no_prior_t2d.value_counts())
print(delivery_df_t3.gest_hyper_no_prior_hyper.value_counts())
print(delivery_df_t3.preeclampsia_no_prior_hyper.value_counts())

In [ ]:
# --- exclude stillbirths --------------------------------------------------
delivery_df_t3 = delivery_df_t3[~delivery_df_t3["PersonId"].isin(stillbirth["PersonId"])]
full_med = full_med[~full_med["PersonId"].isin(stillbirth["PersonId"])]
print("after stillbirth exclusion:", delivery_df_t3["PersonId"].nunique(), full_med["PersonId"].nunique())

## 12. Demographics: age, race/ethnicity, income

In [ ]:
df = ps.sql("SELECT * FROM Person").to_pandas()
person = decode_concepts(df).rename(columns={"Id": "PersonId"})

df = ps.sql("SELECT * FROM PersonRace").to_pandas()
race = decode_concepts(df)

delivery_df_t3 = (delivery_df_t3
                  .merge(person[["PersonId", "BirthDateTime", "Ethnicity", "Gender"]], on="PersonId", how="left")
                  .merge(race[["PersonId", "Race"]], on="PersonId", how="left"))
full_med = (full_med
            .merge(person[["PersonId", "BirthDateTime", "Ethnicity", "Gender"]], on="PersonId", how="left")
            .merge(race[["PersonId", "Race"]], on="PersonId", how="left"))

In [ ]:
def calculate_age(event_date, dob):
    """Age in years between a date of birth and an event date."""
    return (event_date - dob).days / 365.25


for frame in (delivery_df_t3, full_med):
    for col in ["delivery_date", "BirthDateTime", "event_date"]:
        frame[col] = pd.to_datetime(frame[col]).dt.date
    frame["age_at_delivery"] = frame.apply(
        lambda row: calculate_age(row["delivery_date"], row["BirthDateTime"]), axis=1)
    frame["age_at_event"] = frame.apply(
        lambda row: calculate_age(row["event_date"], row["BirthDateTime"]), axis=1)

print("median age at delivery:", delivery_df_t3.age_at_delivery.median())
print("median age at exposure:", delivery_df_t3.age_at_event.median())

In [ ]:
def combine_race_ethnicity(row):
    """Collapse Truveta `Race` + `Ethnicity` into a single analysis variable."""
    race = str(row["Race"]).strip().lower()
    ethnicity = str(row["Ethnicity"]).strip().lower()

    if pd.isna(race) or pd.isna(ethnicity):
        return "Unknown"
    if ethnicity == "hispanic or latino":
        return "Hispanic"
    if ethnicity == "not hispanic or latino":
        if race == "white":
            return "Non-Hispanic White"
        if race == "black or african american":
            return "Non-Hispanic Black"
        if race in ("asian", "american indian or alaska native",
                    "native hawaiian or other pacific islander", "other race"):
            return "Other"
        return "Unknown"
    return "Unknown"


delivery_df_t3["race_ethnicity"] = delivery_df_t3.apply(combine_race_ethnicity, axis=1)
full_med["race_ethnicity"] = full_med.apply(combine_race_ethnicity, axis=1)
delivery_df_t3.race_ethnicity.value_counts()

### Income from Social Determinants of Health

The SDOH table is long-format (one row per person/attribute/date). The SQL below
keeps the most recent value per person per attribute and pivots to wide format;
we use `EstimatedAnnualIncome`.

In [ ]:
snapshot.load_sql_table("SELECT * FROM SocialDeterminantsOfHealth",
                        cache=True, view_name="tbl_sdoh")
snapshot.load_sql_table("SELECT ConceptId, ConceptName FROM tbl_concept",
                        cache=True, view_name="tbl_concept_sel")

# Attach the attribute name to each SDOH row.
snapshot.load_sql_table("""
    SELECT PersonId, EffectiveStartDateTime, AttributeConceptId,
           NormalizedValueNumeric, NormalizedValueConceptId
    FROM tbl_sdoh
""", cache=True, view_name="tbl_sdoh_2")

snapshot.load_sql_table("""
    SELECT s.PersonId, s.EffectiveStartDateTime, s.NormalizedValueNumeric,
           s.NormalizedValueConceptId, c.ConceptId, c.ConceptName AS Attribute
    FROM tbl_sdoh_2 s
    LEFT JOIN tbl_concept_sel c ON s.AttributeConceptId = c.ConceptId
""", cache=True, view_name="tbl_sdoh_3")

# Decode the value concept too.
snapshot.load_sql_table("""
    SELECT s.PersonId, s.EffectiveStartDateTime, s.Attribute,
           c.ConceptName AS Attribute_Value_Categorical,
           s.NormalizedValueNumeric AS Attribute_Value_Numeric
    FROM tbl_sdoh_3 s
    LEFT JOIN tbl_concept_sel c ON s.NormalizedValueConceptId = c.ConceptId
""", cache=True, view_name="tbl_sdoh_4")

# Keep only the most recent record per person per attribute.
SDOH_5 = snapshot.load_sql_table("""
    WITH SortedData AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY PersonId, Attribute
                                  ORDER BY EffectiveStartDateTime DESC, Attribute) AS rn
        FROM tbl_sdoh_4
    )
    SELECT * FROM SortedData WHERE rn = 1
    ORDER BY PersonId, EffectiveStartDateTime, Attribute
""", cache=True, view_name="tbl_sdoh_5")

SDOH_5.head(10)

In [ ]:
SDOH_wide = (SDOH_5.to_pandas()
             .pivot(index="PersonId", columns="Attribute",
                    values=["EffectiveStartDateTime", "Attribute_Value_Categorical",
                            "Attribute_Value_Numeric"])
             .reset_index())
SDOH_wide.columns = SDOH_wide.columns.map(lambda index: f"{index[0]}_{index[1]}")
SDOH_wide.rename({"PersonId_": "PersonId"}, axis=1, inplace=True)

person_SDOH = person.merge(SDOH_wide, how="left", on="PersonId")
person_SDOH.Attribute_Value_Categorical_EstimatedAnnualIncome.value_counts()

In [ ]:
INCOME_COL = "Attribute_Value_Categorical_EstimatedAnnualIncome"


def reclassify_income(bracket):
    """Collapse the SDOH annual-income bracket string into three analysis bands."""
    try:
        low = int(str(bracket).split("-")[0].replace(",", "").strip())
    except (ValueError, AttributeError):
        return "Unknown"
    if low <= 50000:
        return "≤50000"
    if low <= 80000:
        return "50001-80000"
    return ">80000"


for name in ["delivery_df_t3", "full_med"]:
    frame = globals()[name]
    frame = frame.merge(person_SDOH[["PersonId", INCOME_COL]], on="PersonId", how="left")
    frame[INCOME_COL] = frame[INCOME_COL].fillna("Unknown")
    frame["Income"] = frame[INCOME_COL].apply(reclassify_income)
    globals()[name] = frame

delivery_df_t3.Income.value_counts()

In [ ]:
# Weight change between treatment initiation and conception.
delivery_df_t3["weight_loss"] = (
    delivery_df_t3["pretreatment_weight"] - delivery_df_t3["prepreg_weight"])
delivery_df_t3["Race"] = delivery_df_t3["Race"].fillna("Unknown")

# Attach pre-treatment BMI.
delivery_df_t3 = delivery_df_t3.merge(pretreat_df, on="PersonId", how="left")

### Sub-analysis flag: exposure started within one year of conception

In [ ]:
med_delivery_df_t3 = delivery_df_t3[delivery_df_t3.source_type == "med"].copy()
print("medication cohort:", len(med_delivery_df_t3))


def check_event_in_past_year(row, df):
    """True if this person had an exposure date in the year before the LMP."""
    start = row["estimated_LMP"] - pd.DateOffset(years=1)
    end = row["estimated_LMP"]
    person_events = df[df["PersonId"] == row["PersonId"]]["event_date"]
    return ((person_events >= start) & (person_events < end)).any()


med_delivery_df_t3["event_in_past_year"] = med_delivery_df_t3.apply(
    lambda row: check_event_in_past_year(row, med_delivery_df_t3), axis=1)
med_delivery_df_t3.event_in_past_year.value_counts()

## 13. Export

In [ ]:
delivery_df_t3.to_csv(output_path_local + OUT_COHORT, index=False)
all_weights_df_pd.to_csv(output_path_local + OUT_WEIGHTS, index=False)
medication_full.to_csv(output_path_local + OUT_MEDICATION, index=False)
full_med.to_csv(output_path_local + OUT_MED_NOWEIGHT, index=False)

print("wrote:")
for f in (OUT_COHORT, OUT_WEIGHTS, OUT_MEDICATION, OUT_MED_NOWEIGHT, OUT_ZCODE_COUNT):
    print("  ", output_path_local + f)

## 14. Descriptive Table 1 (medication vs surgery)

Descriptive only - the matched, adjusted comparisons live in notebook 05.

In [ ]:
!pip install tableone
from tableone import TableOne

In [ ]:
table1 = pd.read_csv(output_path_local + OUT_COHORT)

T1_COLUMNS = [
    "preterm", "prepreg_weight", "predelivery_weight", "pretreatment_weight",
    "months_between_event_and_conception", "weight_loss", "gestation_weight",
    "PrePregnancyBMI", "nearDeliveryBMI", "preTreatmentBMI", "preeclampsia",
    "csection", "excessive_fetal_weight", "intra_grow_restrict", "obstetric_care",
    "prior_Csection", "Prior_Preterm_Birth", "hyperlipid", "osa", "depression",
    "delivery_type", "infant_gest_age_class", "t2d_before_pregnancy",
    "hyper_before_pregnancy", "parity", "gest_diabetes_no_prior_t2d",
    "gest_hyper_no_prior_hyper", "preeclampsia_no_prior_hyper",
    "race_ethnicity", "age_at_delivery", "age_at_event", "Income",
]

T1_CATEGORICAL = [
    "preterm", "preeclampsia", "csection", "excessive_fetal_weight",
    "intra_grow_restrict", "obstetric_care", "prior_Csection", "Prior_Preterm_Birth",
    "hyperlipid", "osa", "depression", "delivery_type", "infant_gest_age_class",
    "t2d_before_pregnancy", "hyper_before_pregnancy", "parity",
    "gest_diabetes_no_prior_t2d", "gest_hyper_no_prior_hyper",
    "preeclampsia_no_prior_hyper", "race_ethnicity", "Income",
]

table1_p = TableOne(table1, columns=T1_COLUMNS, categorical=T1_CATEGORICAL,
                    groupby="source_type", pval=True)
table1_p.to_html(output_path_local + "/results/table1pvalue.html", index=True)
table1_p

In [ ]:
# Medication group only, restricted to people without pre-pregnancy T2D.
med_not2d = med_delivery_df_t3[med_delivery_df_t3.t2d_before_pregnancy == False].copy()

table1_med = TableOne(med_not2d,
                      columns=T1_COLUMNS + ["event_in_past_year"],
                      categorical=T1_CATEGORICAL + ["event_in_past_year"],
                      pval=False)
table1_med.to_html(output_path_local + "/results/medtable1_not2d.html", index=True)
table1_med

---

## Appendix - superseded approaches (do not run)

Kept for provenance so the analytic history is auditable. None of these feed the
files exported in section 13.

### A1. Preterm birth from diagnosis codes

Superseded by the Z-code gestational-age definition in section 7
(`preterm = gestational_week < 37`). This version matched a separate
preterm-birth code set to deliveries within +/- 30 days.

In [ ]:
# --- SUPERSEDED - DO NOT RUN ---
# preterm_concode = snapshot.codeset_from_prose(url=CODESETS["preterm_birth"], variable_name="codes")
# preterm_con, _ = load_condition_data(snapshot, code_set=preterm_concode,
#                                      view_name="tbl_index_preterm")
#
# cutoff_date = datetime.strptime(DELIVERY_CUTOFF, "%Y-%m-%d").date()
# preterm_con["RecordedDateTime"] = preterm_con["RecordedDateTime"].dt.date
# preterm_con = preterm_con[preterm_con["RecordedDateTime"] >= cutoff_date].dropna()
# preterm_con = (preterm_con.sort_values(["PersonId", "RecordedDateTime"])
#                .drop_duplicates(subset=["PersonId", "RecordedDateTime"], keep="first")
#                .reset_index(drop=True))
#
# # Deliveries whose own code text says preterm/premature.
# coded = delivery_con.copy()
# coded["is_preterm"] = coded["Code"].apply(is_preterm_related)
# coded = coded[coded.is_preterm]
# coded["RecordedDateTime"] = coded["RecordedDateTime"].dt.date
# coded = coded[coded["RecordedDateTime"] >= cutoff_date].reset_index(drop=True)
#
# preterm_df = pd.concat([preterm_con, coded]).sort_values(
#     ["PersonId", "RecordedDateTime"]).reset_index(drop=True)
#
# def match_preterm(row):
#     """Closest preterm record within 30 days either side of the delivery."""
#     records = preterm_df[
#         (preterm_df["PersonId"] == row["PersonId"]) &
#         (preterm_df["RecordedDateTime"] <= row["RecordedDateTime"] + timedelta(days=30)) &
#         (preterm_df["RecordedDateTime"] >= row["RecordedDateTime"] - timedelta(days=30))]
#     if records.empty:
#         return pd.Series([None, None, False])
#     closest = records.loc[(row["RecordedDateTime"] - records["RecordedDateTime"]).idxmin()]
#     return pd.Series([closest["Code"], closest["RecordedDateTime"], True])
#
# delivery_df[["preterm_code", "preterm_date", "is_preterm"]] = delivery_df.apply(
#     match_preterm, axis=1)

### A2. Conception date estimated from a fixed gestational length

Superseded by the Z-code derived `estimated_LMP`. This assumed 35 weeks for
preterm and 39 weeks for term deliveries, and was used by an earlier version of
the pre-pregnancy comorbidity definitions.

In [ ]:
# --- SUPERSEDED - DO NOT RUN ---
# def estimate_conception_date(row):
#     weeks = 35 if row["is_preterm"] else 39
#     return row["RecordedDateTime"] - pd.Timedelta(weeks=weeks)
#
# delivery_df["estimated_conception_date"] = delivery_df.apply(estimate_conception_date, axis=1)
#
# # Pre-pregnancy T2D / hypertension anchored on estimated_conception_date.
# # Section 11 re-derives both against estimated_LMP and overwrites these values.

### A3. Medication persistence / discontinuation

Moved to notebook **03 - Drug episodes**, which is the maintained version.
The block below was an earlier gap-based calculation kept here only for
comparison.

In [ ]:
# --- SUPERSEDED - see notebook 03 - DO NOT RUN ---
# medication_df = medication_full.copy()
# medication_df["med_date"] = pd.to_datetime(medication_df["med_date"])
# medication_df = medication_df.sort_values(["PersonId", "med_date"])
# medication_df = medication_df.groupby(["PersonId", "med_date", "Code"], as_index=False).agg(
#     {"DaysSupply": "sum"})
# medication_df["EndDate"] = medication_df["med_date"] + pd.to_timedelta(
#     medication_df["DaysSupply"], unit="d")
#
# medication_df["NextStart"] = medication_df.groupby("PersonId")["med_date"].shift(-1)
# medication_df["GapDays"] = (medication_df["NextStart"] - medication_df["EndDate"]).dt.days
#
# person_ranges = medication_df.groupby("PersonId").agg(
#     IndexDate=("med_date", "min"), LastEndDate=("EndDate", "max")).reset_index()
# person_ranges["TotalTime"] = (person_ranges["LastEndDate"] - person_ranges["IndexDate"]).dt.days
#
# long_gaps = medication_df[medication_df["GapDays"] >= 60]
# gap_summary = long_gaps.groupby("PersonId").agg(
#     GapTime=("GapDays", "sum"),
#     Discontinued_60=("GapDays", lambda x: len(x) > 0)).reset_index()
#
# feature_table = person_ranges.merge(gap_summary, on="PersonId", how="left")
# feature_table["GapTime"] = feature_table["GapTime"].fillna(0)
# feature_table["Discontinued_60"] = feature_table["Discontinued_60"].fillna(False)
# feature_table["NetMedicationTime"] = feature_table["TotalTime"] - feature_table["GapTime"]